# DAG Parameterization
The goal of this notebook is to assess which of the graphs (DAGs or PAGs) we generated during our causal structure learning step produces the best predictive model.

In [6]:
# Data manipulation and plotting
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import re
import glob

# Bayesian Networks
from pgmpy.models import LinearGaussianBayesianNetwork

# Metrics
from sklearn.calibration import calibration_curve
from pycalib.metrics import ECE

import xgboost
import shap

## Data preparation

In [7]:
# Load training data
X_train = pd.read_csv('../../Data Pre-processing/X_train_c12_w48_imp.csv', index_col=0)
X_test = pd.read_csv('../../Data Pre-processing/X_test_c12_w48_imp.csv', index_col=0)

print(X_train.shape, X_test.shape)

#Remove features with 0 variance
X_train = X_train.loc[:, X_train.var() >= 0.01]
X_test = X_test.filter(items=X_train.columns) # Keep only columns in X_train
print(X_train.shape, X_test.shape)

# # Impute missing values using Iterative Imputer (Linear Gaussian BN can't handle missing values)
# imputer = IterativeImputer(random_state=42, max_iter=100)
# X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
# X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

X_train_imp = pd.read_csv('../../Data Pre-processing/X_train_c12_w48_mice.csv', index_col=0)
X_test_imp = pd.read_csv('../../Data Pre-processing/X_test_c12_w48_mice.csv', index_col=0)

y_train = pd.read_csv('../../Data Pre-processing/y_train_c12_w48_imp.csv', index_col=[0,1]).groupby('uid').max()
y_test = pd.read_csv('../../Data Pre-processing/y_test_c12_w48_imp.csv', index_col=[0,1]).groupby('uid').max()

# Correlation feature selection
correlation_threshold = 0.3
X_train_corr = X_train_imp.loc[:, X_train_imp.corrwith(y_train['Outcome']).abs() >= correlation_threshold]
print(X_train_corr.shape)

(13054, 990) (3895, 990)
(13054, 811) (3895, 811)
(13054, 113)


# Loading DAGs

In [13]:
#Get all adjacency files
adjacency_files = glob.glob("../DAGs/*_adjacency.csv")

dags = {}
dags['Control'] = nx.from_edgelist([(col, 'Outcome') for col in X_train_imp.columns.str.replace(r'(_.+)?$', '', regex=True).unique().tolist() if col != 'Outcome'], create_using=nx.DiGraph())
dags['Correlation'] = nx.from_edgelist([(col, 'Outcome') for col in X_train_corr.columns.str.replace(r'(_.+)?$', '', regex=True).unique().tolist() if col != 'Outcome'], create_using=nx.DiGraph())
for file in adjacency_files:
    dag_name = re.search(r'../DAGs/(.*)_adjacency.csv', file).group(1)
    dag_name = re.sub(r'(?<![a-zA-Z])x(?![a-zA-Z])', lambda _: '$\\cap$', dag_name)
    dag_name = dag_name.replace(' + ', ' $\\cup$ ')
    dags[dag_name] = nx.DiGraph(pd.read_csv(file, index_col=0))

dags.pop('Golem $\\cap$ PCMB')  # Remove problematic DAG (has 0 nodes associated with Outcome)
list(dags.keys())

['Control',
 'Correlation',
 'Clinician Consensus $\\cup$ Golem',
 'Simplified Clinician Consensus $\\cup$ Simplified PCMB',
 'Simplified Clinician Consensus $\\cup$ Simplified Golem',
 'Clinician Consensus $\\cup$ PCMB',
 'Simplified Golem $\\cup$ Simplified PCMB',
 'Clinician Consensus',
 'Simplified Clinician Consensus',
 'Clinician Consensus $\\cap$ PCMB',
 'Golem',
 'PCMB',
 'Simplified Clinician Consensus $\\cap$ Simplified PCMB',
 'Simplified PCMB',
 'Clinician Consensus $\\cap$ Golem',
 'Simplified Golem',
 'Golem $\\cup$ PCMB',
 'Simplified Golem $\\cap$ Simplified PCMB',
 'Simplified Clinician Consensus $\\cap$ Simplified Golem']

In [18]:
list(dags['Simplified Golem'].nodes())

['Outcome',
 'Weight',
 'SpO2',
 'Creatinine',
 'Peds Coma Score',
 'Blood urea nitrogen',
 'Dopamine',
 'Sodium',
 'PTT',
 'Ventilated',
 'Potassium',
 'DBP']

# Model Training Functions
For each dag we train a Linear Gaussian Bayesian Network, an XGBoost

In [4]:
from MLstatkit import Bootstrapping, Delong_test, Permutation_test
from itertools import combinations

def compare_models(y, y_prob1, y_prob2):
        # DeLong Test for AUROC
        z, p = Delong_test(y.values, y_prob1, y_prob2, return_ci=False, return_auc=False)
        metric_a, metric_b, p_value, benchmark, samples_mean, samples_std = Permutation_test(y.values, y_prob1, y_prob2, metric_str='average_precision')
        return (z, p), (metric_a, metric_b, p_value, benchmark, samples_mean, samples_std)

def performance_report(X, y, model):
    try:
        y_prob = model.predict_proba(X)[:, 1] # Predict on test data
    except AttributeError:
        y_prob = model.predict(X)['Outcome'].to_numpy() # Predict on test data (LGBN fallback: pgmpy 1.x returns DataFrame w/ Outcome conditional mean)
        y_prob = np.where(y_prob > 1, 1, np.where(y_prob < 0, 0, y_prob))

    ## Calculate Performance Metrics with Bootstrapping
    n_bootstraps = 1000
    # Average Precision
    ap, ap_lower, ap_upper = Bootstrapping(y.values, y_prob, metric_str='average_precision', n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    # AUROC
    auroc, auroc_lower, auroc_upper = Bootstrapping(y.values, y_prob, metric_str='roc_auc', n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    # Precision 
    precision, precision_lower, precision_upper = Bootstrapping(y.values, y_prob, metric_str='precision', n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    # Recall
    recall, recall_lower, recall_upper = Bootstrapping(y.values, y_prob, metric_str='recall', n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    # F1 Score
    f1, f1_lower, f1_upper = Bootstrapping(y.values, y_prob, metric_str='f1', n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    # Accuracy
    accuracy, accuracy_lower, accuracy_upper = Bootstrapping(y.values, y_prob, metric_str='accuracy', n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    # ECE
    ece = ECE(y.values, y_prob.reshape(-1, 1), bins=50)

    return {'Average Precision': f"{ap:.2f} ({ap_lower:.2f}, {ap_upper:.2f})",
     'AUROC': f"{auroc:.2f} ({auroc_lower:.2f}, {auroc_upper:.2f})",
     'Precision': f"{precision:.2f} ({precision_lower:.2f}, {precision_upper:.2f})",
     'Recall': f"{recall:.2f} ({recall_lower:.2f}, {recall_upper:.2f})",
     'F1 Score': f"{f1:.2f} ({f1_lower:.2f}, {f1_upper:.2f})",
     'Accuracy': f"{accuracy:.2f} ({accuracy_lower:.2f}, {accuracy_upper:.2f})",
     'ECE': f"{ece:.3f}"}

def train_models(remove_drugs=False, remove_interventions=False):
        results = []
        shap_values = {}
        calibrations = {}
        model_preds = {}
        drugs = ['Midazolam', 'Fentanyl', 'Olanzapine', 'Haloperidol', 
                'Dexmedetomidine', 'Lorazepam', 'Morphine', 'Hydromorphone', 
                'Dopamine', 'Cisatracurium', 'Epinephrine', 'Norepinephrine', 
                'Milrinone', 'Dobutamine']
        

        for dag_name, dag in dags.items():
        
                # Simplified is not in the dag_name, skip
                if 'Simplified' not in dag_name and dag_name not in ['Control', 'Correlation']:
                        continue


                if dag is not None:
                        print(f"Processing {dag_name} with {dag.number_of_nodes()} nodes and {dag.number_of_edges()} edges")

                nodes_in_dag = list(dag.nodes())

                if 'Simplified' in dag_name or dag_name in ['Control', 'Correlation']:
                        if remove_drugs:
                                for drug in drugs:
                                        if drug in nodes_in_dag:
                                                nodes_in_dag.remove(drug)
                
                        if remove_interventions:
                                intervention_nodes = ['CRRT Therapy Type', 'ECMO Type', 'Ventilated', 'Pupillary Reaction', 'Peds Coma Score', 'Endotracheal tube']
                                for intervention in intervention_nodes:
                                        if intervention in nodes_in_dag:
                                                nodes_in_dag.remove(intervention)

                        if dag_name == 'Correlation':
                                features_in_dag = [col for col in X_train_corr.columns if any(re.search(rf'^{re.escape(node)}(_.+)?$', col) for node in nodes_in_dag)]
                        else:
                                features_in_dag = [col for col in X_train_imp.columns if any(re.search(rf'^{re.escape(node)}(_.+)?$', col) for node in nodes_in_dag)]

                        train_dag = nx.DiGraph()

                        # Map DAG nodes to features
                        for node in nodes_in_dag:
                                ends = list(dag.successors(node))
                                from_features = [col for col in features_in_dag + ['Outcome'] if re.search(rf'^{re.escape(node)}(_.+)?$', col)]
                                to_features = [col for col in features_in_dag + ['Outcome'] if any(re.search(rf'^{re.escape(end)}(_.+)?$', col) for end in ends)]
                                for from_feature in from_features:
                                        for to_feature in to_features:
                                                train_dag.add_edge(from_feature, to_feature)
                        
                        n_biomarkers = len(nodes_in_dag) - 1  # Exclude Outcome
                else:
                        # Use list comprehensions to avoid mutating nodes_in_dag while iterating
                        if remove_drugs:
                                nodes_in_dag = [node for node in nodes_in_dag if not any(drug in node for drug in drugs)]
                        if remove_interventions:
                                intervention_nodes = ['CRRT Therapy Type', 'ECMO Type', 'Ventilated', 'Pupillary Reaction', 'Peds Coma Score', 'Endotracheal tube']
                                nodes_in_dag = [node for node in nodes_in_dag if not any(intervention in node for intervention in intervention_nodes)]
                        
                        features_in_dag = X_train_imp.filter(items=nodes_in_dag).columns.tolist()
                        train_dag = dag.subgraph(features_in_dag + ['Outcome']).copy()
                        n_biomarkers = X_train_imp.filter(items=nodes_in_dag).columns.str.replace(r'(_.+)?$', '', regex=True).nunique() - 1  # Exclude Outcome

                # XGB trains on non-imputed X_train; restrict to columns actually present there
                xgb_features = [f for f in features_in_dag if f in X_train.columns]

                models = {}
                # Instantiate Models
                models['LGBN'] = LinearGaussianBayesianNetwork(train_dag.edges())
                # models['LR'] = LogisticRegression(max_iter=1000, random_state=42)
                models['XGB'] = xgboost.XGBClassifier(objective='binary:logistic', random_state=42, eval_metric='aucpr')
                

                for model_name, model in models.items():
                        metrics = {}
                        metrics['Model'] = model_name
                        metrics['DAG'] = dag_name

                        if model_name == 'LGBN':
                                # Fit the model
                                train = pd.concat([X_train_imp.filter(features_in_dag), y_train.filter(['Outcome']).astype(int)], axis=1)
                                model.fit(train)
                                metrics.update(performance_report(X_test_imp.filter(features_in_dag), y_test.Outcome, model))
                                metrics['# Features'] = len(features_in_dag)
                        elif model_name == 'LR':
                                model.fit(X_train_imp.filter(features_in_dag), y_train.Outcome)
                                model_preds[dag_name] = model.predict_proba(X_test_imp.filter(features_in_dag))[:, 1]
                                metrics.update(performance_report(X_test_imp.filter(features_in_dag), y_test.Outcome, model))
                                metrics['# Features'] = len(features_in_dag)
                        else:
                                X_train_filtered = X_train.filter(xgb_features)
                                X_test_filtered = X_test.filter(xgb_features)
                                model.fit(X_train_filtered, y_train.Outcome)
                                y_prob = model.predict_proba(X_test_filtered)[:, 1]
                                model_preds[dag_name] = y_prob
                                metrics.update(performance_report(X_test_filtered, y_test.Outcome, model))
                                calibrations[dag_name] = calibration_curve(y_test.Outcome, y_prob, n_bins=50)
                                metrics['# Features'] = len(xgb_features)

                        metrics['# Biomarkers'] = n_biomarkers

                        results.append(metrics)


                # SHAP feature importance
                X_test_shap = X_test.filter(xgb_features)
                explainer = shap.TreeExplainer(models['XGB'])
                values = explainer.shap_values(X_test_shap)
                values = pd.DataFrame(values, columns=X_test_shap.columns, index=X_test_shap.index)
                shap_values[dag_name] = values
        

        return results, shap_values, calibrations, model_preds

In [5]:
def shap_values_biomarker_summary(shap_values):
    vals = {} 
    for dag_name in shap_values:
        df = shap_values[dag_name].reset_index().melt(id_vars='uid')
        df.variable = df.variable.str.replace(r'(_.+)?$', '', regex=True)
        df.value = df.value.abs()                                   # abs before summing to prevent sign cancellation
        df = df.groupby(['uid', 'variable']).sum().reset_index()
        df = df.groupby('variable').value.mean().sort_values(ascending=True)
        vals[dag_name] = df
    return vals

# Model Training

## Bootstrapping
We simulate the potential performance of our models on an out of distribution dataset by performing a weighted bootstrap of our Test set. <br>
For context, our test set is already based on prospective samples. The training set comes from patients admitted up to 2019 and the test set is from patient admitted on or after 2020.

## Expriment 1: Comparing Feature Selection Approaches (DAGs)

In [50]:
results, shap_values, calibrations, model_preds = train_models(remove_drugs=False, remove_interventions=False)

Processing Control with 46 nodes and 45 edges


Bootstrapping accuracy: 100%|██████████| 1000/1000 [00:00<00:00, 4127.87it/s]


Processing Correlation with 38 nodes and 37 edges


Bootstrapping accuracy: 100%|██████████| 1000/1000 [00:00<00:00, 4244.54it/s]


Processing Simplified Clinician Consensus $\cup$ Simplified PCMB with 19 nodes and 52 edges


...

Bootstrapping accuracy: 100%|██████████| 1000/1000 [00:00<00:00, 4581.33it/s]


Processing Simplified Golem $\cap$ Simplified PCMB with 6 nodes and 5 edges


Bootstrapping accuracy: 100%|██████████| 1000/1000 [00:00<00:00, 4402.69it/s]


Processing Simplified Clinician Consensus $\cap$ Simplified Golem with 5 nodes and 4 edges


Bootstrapping accuracy: 100%|██████████| 1000/1000 [00:00<00:00, 4288.17it/s]


In [51]:
aurocs = []
auprcs = []
print(len(model_preds))
# Model Comparisons using DeLong Test
for (model1_name, model2_name) in combinations(model_preds.keys(), 2):
        if model1_name != 'Control': continue
        model1 = model_preds[model1_name]
        model2 = model_preds[model2_name]
        comparison_result1, comparison_result2 = compare_models(y_test.Outcome, model1, model2)
        aurocs.append({'Model 1': model1_name, 'Model 2': model2_name, 'Z-score': f"{comparison_result1[0]:.4f}", 'P-value': f"{comparison_result1[1]:.3e}", 'p_numeric': comparison_result1[1]})
        auprcs.append({'Model 1': model1_name, 'Model 2': model2_name, 'Avg Precision Model 1': f"{comparison_result2[0]:.4f}", 'Avg Precision Model 2': f"{comparison_result2[1]:.4f}", 'P-value': f"{comparison_result2[2]:.3e}", 'Benchmark': comparison_result2[3], 'Samples Mean': f"{comparison_result2[4]:.4f}", 'Samples Std': f"{comparison_result2[5]:.4f}"})
aurocs_df = pd.DataFrame(aurocs)
auprcs_df = pd.DataFrame(auprcs)
aurocs_df.to_csv('../Predictive Modeling/delong_test_biomarker_selection_auroc.csv', index=False)
auprcs_df.to_csv('../Predictive Modeling/delong_test_biomarker_selection_auprc.csv', index=False)
aurocs_df

11


Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 788.90it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 818.66it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 822.09it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 810.22it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 774.95it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 814.84it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 805.62it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 788.52it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 797.97it/s]
Computing average_precision Permutati

,Model 1,Model 2,Z-score,P-value,p_numeric
0,Control,Correlation,-5.0469,4.491e-07,4.491476e-07
1,Control,Simplified Clinician Consensus $\cup$ Simplifi...,-3.1917,1.415e-03,1.414528e-03
2,Control,Simplified Clinician Consensus $\cup$ Simplifi...,-3.5661,3.623e-04,3.622732e-04
3,Control,Simplified Golem $\cup$ Simplified PCMB,-3.4346,5.934e-04,5.933930e-04
4,Control,Simplified Clinician Consensus,-3.8218,1.325e-04,1.324738e-04
5,Control,Simplified Clinician Consensus $\cap$ Simplifi...,-3.5869,3.346e-04,3.345848e-04
6,Control,Simplified PCMB,-3.7981,1.458e-04,1.457824e-04
7,Control,Simplified Golem,-3.1809,1.468e-03,1.468225e-03
8,Control,Simplified Golem $\cap$ Simplified PCMB,-3.7163,2.022e-04,2.021565e-04
9,Control,Simplified Clinician Consensus $\cap$ Simplifi...,-5.5644,2.630e-08,2.630400e-08


In [52]:
pd.DataFrame(results).to_csv('../Predictive Modeling/Biomarker Selection.csv', index=False)
pd.DataFrame(results)

,Model,DAG,Average Precision,AUROC,Precision,Recall,F1 Score,Accuracy,ECE,# Features,# Biomarkers
0,LGBN,Control,"0.76 (0.72, 0.80)","0.90 (0.88, 0.92)","0.91 (0.89, 0.93)","0.78 (0.76, 0.81)","0.83 (0.81, 0.85)","0.94 (0.94, 0.95)",0.139,811,45
1,XGB,Control,"0.81 (0.78, 0.84)","0.93 (0.91, 0.95)","0.94 (0.92, 0.95)","0.80 (0.78, 0.83)","0.85 (0.83, 0.87)","0.95 (0.94, 0.96)",0.089,811,45
2,LGBN,Correlation,"0.73 (0.69, 0.76)","0.87 (0.84, 0.89)","0.95 (0.93, 0.96)","0.74 (0.72, 0.77)","0.81 (0.79, 0.83)","0.94 (0.93, 0.95)",0.133,113,37
3,XGB,Correlation,"0.75 (0.71, 0.78)","0.90 (0.88, 0.91)","0.90 (0.88, 0.92)","0.78 (0.76, 0.81)","0.83 (0.81, 0.85)","0.94 (0.94, 0.95)",0.108,113,37
4,LGBN,Simplified Clinician Consensus $\cup$ Simplifi...,"0.76 (0.72, 0.79)","0.91 (0.89, 0.93)","0.93 (0.92, 0.95)","0.77 (0.74, 0.79)","0.83 (0.80, 0.85)","0.94 (0.94, 0.95)",0.136,315,18
5,XGB,Simplified Clinician Consensus $\cup$ Simplifi...,"0.79 (0.75, 0.82)","0.91 (0.89, 0.93)","0.92 (0.90, 0.93)","0.80 (0.78, 0.83)","0.85 (0.83, 0.87)","0.95 (0.94, 0.96)",0.094,315,18
6,LGBN,Simplified Clinician Consensus $\cup$ Simplifi...,"0.74 (0.70, 0.78)","0.90 (0.88, 0.92)","0.93 (0.92, 0.95)","0.77 (0.75, 0.79)","0.83 (0.81, 0.85)","0.94 (0.94, 0.95)",0.140,389,22
7,XGB,Simplified Clinician Consensus $\cup$ Simplifi...,"0.78 (0.75, 0.82)","0.91 (0.89, 0.93)","0.92 (0.90, 0.94)","0.79 (0.77, 0.82)","0.84 (0.82, 0.86)","0.95 (0.94, 0.95)",0.092,389,22
8,LGBN,Simplified Golem $\cup$ Simplified PCMB,"0.75 (0.71, 0.79)","0.91 (0.89, 0.92)","0.93 (0.92, 0.95)","0.77 (0.75, 0.80)","0.83 (0.81, 0.85)","0.94 (0.94, 0.95)",0.141,407,23
9,XGB,Simplified Golem $\cup$ Simplified PCMB,"0.78 (0.75, 0.81)","0.91 (0.89, 0.93)","0.91 (0.89, 0.93)","0.79 (0.77, 0.81)","0.84 (0.82, 0.86)","0.94 (0.94, 0.95)",0.093,407,23


In [ ]:
# Plot calibration curves
for dag_name in calibrations:
    plt.figure(dpi=300, layout='constrained', figsize=(8, 5))
    plt.rcParams['font.size'] = 18
    prob_true, prob_pred = calibrations[dag_name]
    plt.plot(prob_pred, prob_true, marker='o', label='Calibration Curve')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
    plt.xlabel('Predicted Probability')
    plt.ylabel('True Probability')
    plt.xticks(np.arange(0, 1.1, 0.1))
    plt.grid()
    plt.savefig(f'../Predictive Modeling/Calibration Curves/Biomarker Selection - {dag_name.replace(" ", "_").replace("$\\cap$", "AND").replace("$\\cup$", "OR")}.pdf', bbox_inches='tight', dpi=300)
    plt.close()
plt.rcParams.update(plt.rcParamsDefault)

In [ ]:
vals = shap_values_biomarker_summary(shap_values)
auprc_top10 = pd.DataFrame(results).sort_values(by=['Average Precision'], ascending=False).head(10).DAG.unique().tolist()
plt.rcParams.update({  
'figure.figsize': (8, 10),   # Default figure size (inches)  
'figure.dpi': 300,          # Default DPI for figure creation  
'ytick.labelsize': 16,
'xtick.labelsize': 16,
'font.size': 18,
'lines.linewidth': 2,       # Default line width  
'figure.constrained_layout.use': True,
'pdf.fonttype': 42
}) 
for dag_name in auprc_top10:
    df = vals[dag_name].sort_values(ascending=True)
    plt.barh(df.index, df)
    plt.xlabel('Mean |SHAP value|')
    plt.ylabel('Biomarker')
    plt.savefig(f'../Predictive Modeling/Feature Importance/Biomarker Selection - {dag_name}.pdf', bbox_inches='tight', dpi=300)
    plt.close()
plt.rcParams.update(plt.rcParamsDefault)

## Experiment 2: No Drugs

In [55]:
results, shap_values, calibrations, model_preds = train_models(remove_drugs=True, remove_interventions=False)

Processing Control with 46 nodes and 45 edges


Bootstrapping accuracy: 100%|██████████| 1000/1000 [00:00<00:00, 4433.90it/s]


Processing Correlation with 38 nodes and 37 edges


Bootstrapping accuracy: 100%|██████████| 1000/1000 [00:00<00:00, 4447.55it/s]


Processing Simplified Clinician Consensus $\cup$ Simplified PCMB with 19 nodes and 52 edges


...

Bootstrapping accuracy: 100%|██████████| 1000/1000 [00:00<00:00, 4643.37it/s]


Processing Simplified Golem $\cap$ Simplified PCMB with 6 nodes and 5 edges


Bootstrapping accuracy: 100%|██████████| 1000/1000 [00:00<00:00, 4593.12it/s]


Processing Simplified Clinician Consensus $\cap$ Simplified Golem with 5 nodes and 4 edges


Bootstrapping accuracy: 100%|██████████| 1000/1000 [00:00<00:00, 4453.05it/s]


In [56]:
pd.DataFrame(results).to_csv('../Predictive Modeling/No Drugs.csv', index=False)
pd.DataFrame(results)

,Model,DAG,Average Precision,AUROC,Precision,Recall,F1 Score,Accuracy,ECE,# Features,# Biomarkers
0,LGBN,Control,"0.76 (0.71, 0.80)","0.90 (0.88, 0.92)","0.92 (0.91, 0.94)","0.78 (0.76, 0.80)","0.83 (0.81, 0.85)","0.94 (0.94, 0.95)",0.137,624,34
1,XGB,Control,"0.80 (0.77, 0.83)","0.92 (0.91, 0.94)","0.92 (0.90, 0.94)","0.80 (0.78, 0.82)","0.85 (0.83, 0.87)","0.95 (0.94, 0.96)",0.090,624,34
2,LGBN,Correlation,"0.72 (0.68, 0.76)","0.86 (0.84, 0.89)","0.94 (0.93, 0.96)","0.73 (0.71, 0.75)","0.80 (0.77, 0.82)","0.94 (0.93, 0.95)",0.129,70,26
3,XGB,Correlation,"0.75 (0.71, 0.79)","0.89 (0.87, 0.91)","0.90 (0.88, 0.92)","0.78 (0.76, 0.80)","0.82 (0.80, 0.85)","0.94 (0.93, 0.95)",0.111,70,26
4,LGBN,Simplified Clinician Consensus $\cup$ Simplifi...,"0.75 (0.71, 0.78)","0.90 (0.88, 0.92)","0.93 (0.91, 0.95)","0.75 (0.73, 0.78)","0.81 (0.79, 0.83)","0.94 (0.93, 0.95)",0.133,198,11
5,XGB,Simplified Clinician Consensus $\cup$ Simplifi...,"0.77 (0.74, 0.81)","0.91 (0.89, 0.93)","0.91 (0.89, 0.93)","0.79 (0.77, 0.81)","0.84 (0.81, 0.86)","0.94 (0.94, 0.95)",0.100,198,11
6,LGBN,Simplified Clinician Consensus $\cup$ Simplifi...,"0.74 (0.70, 0.78)","0.90 (0.88, 0.92)","0.93 (0.91, 0.94)","0.76 (0.73, 0.78)","0.82 (0.79, 0.84)","0.94 (0.93, 0.95)",0.137,271,15
7,XGB,Simplified Clinician Consensus $\cup$ Simplifi...,"0.77 (0.74, 0.81)","0.91 (0.89, 0.93)","0.91 (0.89, 0.92)","0.80 (0.78, 0.82)","0.84 (0.82, 0.86)","0.95 (0.94, 0.95)",0.099,271,15
8,LGBN,Simplified Golem $\cup$ Simplified PCMB,"0.74 (0.70, 0.78)","0.90 (0.88, 0.92)","0.92 (0.90, 0.94)","0.75 (0.73, 0.78)","0.81 (0.79, 0.83)","0.94 (0.93, 0.95)",0.138,271,15
9,XGB,Simplified Golem $\cup$ Simplified PCMB,"0.78 (0.75, 0.82)","0.91 (0.89, 0.93)","0.91 (0.89, 0.93)","0.79 (0.77, 0.82)","0.84 (0.82, 0.86)","0.95 (0.94, 0.95)",0.096,271,15


In [57]:
aurocs = []
auprcs = []
print(len(model_preds))
# Model Comparisons using DeLong Test
for (model1_name, model2_name) in combinations(model_preds.keys(), 2):
        model1 = model_preds[model1_name]
        model2 = model_preds[model2_name]
        comparison_result1, comparison_result2 = compare_models(y_test.Outcome, model1, model2)
        aurocs.append({'Model 1': model1_name, 'Model 2': model2_name, 'Z-score': f"{comparison_result1[0]:.4f}", 'P-value': f"{comparison_result1[1]:.3e}", 'p_numeric': comparison_result1[1]})
        auprcs.append({'Model 1': model1_name, 'Model 2': model2_name, 'Avg Precision Model 1': f"{comparison_result2[0]:.4f}", 'Avg Precision Model 2': f"{comparison_result2[1]:.4f}", 'P-value': f"{comparison_result2[2]:.3e}", 'Benchmark': comparison_result2[3], 'Samples Mean': f"{comparison_result2[4]:.4f}", 'Samples Std': f"{comparison_result2[5]:.4f}"})
aurocs_df = pd.DataFrame(aurocs)
auprcs_df = pd.DataFrame(auprcs)
aurocs_df.to_csv('../Predictive Modeling/delong_test_no_drugs_auroc.csv', index=False)
auprcs_df.to_csv('../Predictive Modeling/delong_test_no_drugs_auprc.csv', index=False)

11


Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 769.40it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 797.06it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 798.72it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 782.00it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 784.16it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 811.51it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 800.44it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 804.16it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 816.00it/s]
Computing average_precision Permutati

In [ ]:
# Plot calibration curves
for dag_name in calibrations:
    plt.figure(dpi=300, layout='constrained', figsize=(8, 5))
    plt.rcParams['font.size'] = 18
    prob_true, prob_pred = calibrations[dag_name]
    plt.plot(prob_pred, prob_true, marker='o', label=f'Calibration Curve')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
    plt.xlabel('Predicted Probability')
    plt.ylabel('True Probability')
    # plt.title(f'No Drugs - {dag_name}')
    plt.xticks(np.arange(0, 1.1, 0.1))
    plt.grid()
    plt.savefig(f'../Predictive Modeling/Calibration Curves/No Drugs - {dag_name.replace(" ", "_").replace("$\\cap$", "AND").replace("$\\cup$", "OR")}.pdf', bbox_inches='tight', dpi=300)
    plt.close()
plt.rcParams.update(plt.rcParamsDefault)

In [ ]:
vals = shap_values_biomarker_summary(shap_values)
auprc_top10 = pd.DataFrame(results).sort_values(by=['Average Precision'], ascending=False).head(10).DAG.unique().tolist()

plt.rcParams.update({  
'figure.figsize': (8, 10),   # Default figure size (inches)  
'figure.dpi': 300,          # Default DPI for figure creation  
'ytick.labelsize': 16,
'xtick.labelsize': 16,
'font.size': 18,
'lines.linewidth': 2,       # Default line width  
'figure.constrained_layout.use': True,
'pdf.fonttype': 42
}) 
for dag_name in auprc_top10:
    df = vals[dag_name].sort_values(ascending=True)
    plt.barh(df.index, df)
    plt.xlabel('Mean |SHAP value|')
    plt.ylabel('Biomarker')
    plt.savefig(f'../Predictive Modeling/Feature Importance/No Drugs - {dag_name}.pdf', bbox_inches='tight', dpi=300, transparent=False)
    plt.close()
plt.rcParams.update(plt.rcParamsDefault)

## Experiment 3: No Drugs or Interventions

In [ ]:
results, shap_values, calibrations, model_preds = train_models(remove_drugs=True, remove_interventions=True)

In [ ]:
pd.DataFrame(results).to_csv('../Predictive Modeling/No Drugs or Interventions.csv', index=False)
pd.DataFrame(results)

,Model,DAG,Average Precision,AUROC,Precision,Recall,F1 Score,Accuracy,ECE,# Features,# Biomarkers
0,LGBN,Control,"0.75 (0.71, 0.79)","0.90 (0.87, 0.92)","0.92 (0.90, 0.94)","0.77 (0.75, 0.80)","0.83 (0.81, 0.85)","0.94 (0.94, 0.95)",0.137,521,28
1,XGB,Control,"0.78 (0.75, 0.82)","0.92 (0.90, 0.93)","0.90 (0.88, 0.92)","0.80 (0.78, 0.82)","0.84 (0.82, 0.86)","0.95 (0.94, 0.95)",0.093,521,28
2,LGBN,Correlation,"0.69 (0.66, 0.74)","0.85 (0.82, 0.87)","0.94 (0.93, 0.96)","0.72 (0.70, 0.74)","0.78 (0.76, 0.81)","0.94 (0.93, 0.94)",0.130,53,20
3,XGB,Correlation,"0.75 (0.71, 0.78)","0.89 (0.87, 0.91)","0.90 (0.88, 0.92)","0.78 (0.76, 0.80)","0.83 (0.80, 0.85)","0.94 (0.93, 0.95)",0.114,53,20
4,LGBN,Simplified Clinician Consensus $\cup$ Simplifi...,"0.69 (0.65, 0.74)","0.87 (0.85, 0.90)","0.91 (0.89, 0.93)","0.73 (0.70, 0.75)","0.78 (0.76, 0.81)","0.93 (0.93, 0.94)",0.145,131,7
5,XGB,Simplified Clinician Consensus $\cup$ Simplifi...,"0.76 (0.73, 0.80)","0.91 (0.89, 0.92)","0.89 (0.87, 0.91)","0.79 (0.77, 0.81)","0.83 (0.81, 0.85)","0.94 (0.93, 0.95)",0.104,131,7
6,LGBN,Simplified Clinician Consensus $\cup$ Simplifi...,"0.72 (0.68, 0.75)","0.88 (0.86, 0.90)","0.90 (0.88, 0.93)","0.73 (0.71, 0.75)","0.79 (0.76, 0.81)","0.93 (0.93, 0.94)",0.142,204,11
7,XGB,Simplified Clinician Consensus $\cup$ Simplifi...,"0.77 (0.74, 0.81)","0.91 (0.90, 0.93)","0.91 (0.89, 0.93)","0.79 (0.77, 0.81)","0.84 (0.82, 0.86)","0.95 (0.94, 0.95)",0.101,204,11
8,LGBN,Simplified Golem $\cup$ Simplified PCMB,"0.72 (0.68, 0.76)","0.88 (0.86, 0.90)","0.91 (0.89, 0.93)","0.73 (0.71, 0.76)","0.79 (0.77, 0.81)","0.93 (0.93, 0.94)",0.142,223,12
9,XGB,Simplified Golem $\cup$ Simplified PCMB,"0.77 (0.73, 0.80)","0.91 (0.89, 0.93)","0.91 (0.89, 0.93)","0.78 (0.76, 0.81)","0.83 (0.81, 0.85)","0.94 (0.94, 0.95)",0.098,223,12


In [62]:
aurocs = []
auprcs = []
print(len(model_preds))
# Model Comparisons using DeLong Test
for (model1_name, model2_name) in combinations(model_preds.keys(), 2):
        model1 = model_preds[model1_name]
        model2 = model_preds[model2_name]
        comparison_result1, comparison_result2 = compare_models(y_test.Outcome, model1, model2)
        aurocs.append({'Model 1': model1_name, 'Model 2': model2_name, 'Z-score': f"{comparison_result1[0]:.4f}", 'P-value': f"{comparison_result1[1]:.3e}", 'p_numeric': comparison_result1[1]})
        auprcs.append({'Model 1': model1_name, 'Model 2': model2_name, 'Avg Precision Model 1': f"{comparison_result2[0]:.4f}", 'Avg Precision Model 2': f"{comparison_result2[1]:.4f}", 'P-value': f"{comparison_result2[2]:.3e}", 'Benchmark': comparison_result2[3], 'Samples Mean': f"{comparison_result2[4]:.4f}", 'Samples Std': f"{comparison_result2[5]:.4f}"})
aurocs_df = pd.DataFrame(aurocs)
auprcs_df = pd.DataFrame(auprcs)
aurocs_df.to_csv('../Predictive Modeling/delong_test_only_vitals_labs_auroc.csv', index=False)
auprcs_df.to_csv('../Predictive Modeling/delong_test_only_vitals_labs_auprc.csv', index=False)

11


Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 744.26it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 766.78it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 775.51it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 774.32it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 763.76it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 773.86it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 792.26it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 798.02it/s]
Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 782.39it/s]
Computing average_precision Permutati

In [ ]:
# Plot calibration curves
for dag_name in calibrations:
    plt.figure(dpi=300, layout='constrained', figsize=(8, 5))
    plt.rcParams['font.size'] = 18
    prob_true, prob_pred = calibrations[dag_name]
    plt.plot(prob_pred, prob_true, marker='o', label=f'Calibration Curve')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
    plt.xlabel('Predicted Probability')
    plt.ylabel('True Probability')
    # plt.title(f'Only vitals and labs - {dag_name}')
    # plt.legend()
    plt.xticks(np.arange(0, 1.1, 0.1))
    plt.grid()
    plt.savefig(f'../Predictive Modeling/Calibration Curves/Only vitals and labs - {dag_name.replace(" ", "_").replace("$\\cap$", "AND").replace("$\\cup$", "OR")}.pdf', bbox_inches='tight', dpi=300)
    plt.close()
plt.rcParams.update(plt.rcParamsDefault)

In [64]:
vals = shap_values_biomarker_summary(shap_values)
auprc_top10 = pd.DataFrame(results).sort_values(by=['Average Precision'], ascending=False).head(10).DAG.unique().tolist()
plt.rcParams.update({  
'figure.figsize': (8, 10),   # Default figure size (inches)  
'figure.dpi': 300,          # Default DPI for figure creation  
'ytick.labelsize': 16,
'xtick.labelsize': 16,
'font.size': 18,
'lines.linewidth': 2,       # Default line width  
'figure.constrained_layout.use': True,
'pdf.fonttype': 42
}) 
for dag_name in auprc_top10:
    df = vals[dag_name].sort_values(ascending=True)
    plt.barh(df.index, df)
    plt.xlabel('Mean |SHAP value|')
    plt.ylabel('Biomarker')
    plt.savefig(f'../Predictive Modeling/Feature Importance/Only vitals and labs - {dag_name}.pdf', bbox_inches='tight', dpi=300)
    plt.close()
plt.rcParams.update(plt.rcParamsDefault)

INFO:fontTools.subset:maxp pruned
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:kern dropped
INFO:fontTools.subset:post pruned
INFO:fontTools.subset:FFTM dropped
INFO:fontTools.subset:GPOS pruned
INFO:fontTools.subset:GSUB pruned
INFO:fontTools.subset:glyf pruned
INFO:fontTools.subset:Added gid0 to subset
INFO:fontTools.subset:Added first four glyphs to subset
INFO:fontTools.subset:Closing glyph list over 'MATH': 52 glyphs before
INFO:fontTools.subset:Glyph names: ['.notdef', '.null', 'A', 'B', 'C', 'D', 'G', 'H', 'I', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'W', 'a', 'b', 'bar', 'c', 'd', 'e', 'f', 'five', 'four', 'g', 'h', 'hyphen', 'i', 'k', 'l', 'm', 'n', 'nonmarkingreturn', 'o', 'one', 'p', 'period', 'r', 's', 'six', 'space', 't', 'three', 'two', 'u', 'v', 'x', 'y', 'zero']
INFO:fontTools.subset:Glyph IDs:   [0, 1, 2, 3, 16, 17, 19, 20, 21, 22, 23, 24, 25, 36, 37, 38, 39, 42, 43, 44, 47, 48, 49, 50, 51, 53, 54, 55, 58, 68, 69, 70, 71, 72, 73, 74, 75, 76, 78, 79, 80, 81, 